# 🏥 Donor Dataset — Full Preprocessing Pipeline
> 800 Donors · donors(opal).csv → donor_preprocessed_FINAL.xlsx

In [1]:
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print("✔ Libraries loaded")

✔ Libraries loaded


## STEP 0 — Load Raw Data

In [3]:
df = pd.read_csv(r"E:\final datset of opal ai\dataset\donors(opal).csv")
print(f"[Step 0] Raw data loaded: {df.shape[0]} rows × {df.shape[1]} cols")
print("Columns:", df.columns.tolist())

[Step 0] Raw data loaded: 800 rows × 16 cols
Columns: ['donor_id', 'first_name', 'last_name', 'age', 'gender', 'blood_type', 'organs_donating', 'medical_conditions', 'hepatitis_status', 'alive/deceased', 'time_of_death', 'cause_of_death', 'time_harvested', 'contact_number', 'city', 'hospital_name']


## STEP 1 — Drop PII / Unnecessary Columns

In [4]:
pii_cols = ["donor_id", "first_name", "last_name", "contact_number"]
df.drop(columns=[c for c in pii_cols if c in df.columns], inplace=True)
print(f"[Step 1] Dropped PII columns. Shape: {df.shape}")

[Step 1] Dropped PII columns. Shape: (800, 12)


## STEP 2 — Standardise Text Columns

In [5]:
str_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in str_cols:
    df[col] = df[col].str.strip()

# Standardise organ names
normalization_map = {"Kidneys": "Kidney", "Corneas": "Cornea", "Lungs": "Lung", "Platelets": "Platelet"}
if "organs_donating" in df.columns:
    df["organs_donating"] = df["organs_donating"].apply(
        lambda x: ";".join([normalization_map.get(o.strip(), o.strip()) for o in str(x).split(";")]) if pd.notna(x) else x
    )
print("[Step 2] Standardised text columns and organ names")

[Step 2] Standardised text columns and organ names


C:\Users\T480\AppData\Local\Temp\ipykernel_18828\2907285780.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=["object"]).columns.tolist()


## STEP 3 — DateTime Feature Extraction

In [6]:
df["time_of_death"]  = pd.to_datetime(df["time_of_death"],  errors="coerce")
df["time_harvested"] = pd.to_datetime(df["time_harvested"], errors="coerce")

df["death_hour"]  = df["time_of_death"].dt.hour.fillna(-1).astype(int)
df["death_day"]   = df["time_of_death"].dt.dayofweek.fillna(-1).astype(int)  # 0=Mon
df["death_month"] = df["time_of_death"].dt.month.fillna(-1).astype(int)

# Harvest interval in hours
df["harvest_time_hours"] = (df["time_harvested"] - df["time_of_death"]).dt.total_seconds() / 3600
df["harvest_time_hours"] = df["harvest_time_hours"].fillna(-1)

df.drop(columns=["time_of_death", "time_harvested"], inplace=True)
print("[Step 3] Extracted datetime features: death_hour, death_day, death_month, harvest_time_hours")

[Step 3] Extracted datetime features: death_hour, death_day, death_month, harvest_time_hours


## STEP 4 — Handle Missing Values

In [7]:
df["medical_conditions"] = df["medical_conditions"].fillna("None")
df["cause_of_death"]    = df["cause_of_death"].fillna("Unknown")
df["blood_type"]        = df["blood_type"].fillna("Unknown").str.strip().str.upper()
print("[Step 4] Filled missing values:")
print("         medical_conditions NaN → 'None'")
print("         cause_of_death     NaN → 'Unknown'")
print("         blood_type         NaN → 'Unknown'")
print(f"\nMissing values after fill:")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else "  None ✓")

[Step 4] Filled missing values:
         medical_conditions NaN → 'None'
         cause_of_death     NaN → 'Unknown'
         blood_type         NaN → 'Unknown'

Missing values after fill:
  None ✓


## STEP 5 — Encode Binary Columns

In [8]:
# Gender
df["gender"] = df["gender"].str.strip().str.lower().replace(
    {"male":"Male","m":"Male","female":"Female","f":"Female"})
df["gender_Male"] = (df["gender"] == "Male").astype(int)
df.drop(columns=["gender"], inplace=True)

# Alive / Deceased
df["alive/deceased"] = df["alive/deceased"].map({"Alive":1,"Deceased":0})

# Hepatitis Status
df["hepatitis_status"] = df["hepatitis_status"].map({"Negative":0,"Positive":1,"Unknown":0})

print("[Step 5] Binary encoded:")
print("         gender → gender_Male (1=Male, 0=Female)")
print("         alive/deceased → 1=Alive, 0=Deceased")
print("         hepatitis_status → 1=Positive, 0=Negative")

[Step 5] Binary encoded:
         gender → gender_Male (1=Male, 0=Female)
         alive/deceased → 1=Alive, 0=Deceased
         hepatitis_status → 1=Positive, 0=Negative


## STEP 6 — One-Hot Encode Blood Type

In [9]:
valid_blood = ["A+","A-","B+","B-","O+","O-","AB+","AB-"]
df.loc[~df["blood_type"].isin(valid_blood), "blood_type"] = "Unknown"
blood_dummies = pd.get_dummies(df["blood_type"], prefix="blood").astype(int)
df = pd.concat([df.drop(columns=["blood_type"]), blood_dummies], axis=1)
print(f"[Step 6] One-hot encoded blood_type → {list(blood_dummies.columns)}")

[Step 6] One-hot encoded blood_type → ['blood_A+', 'blood_A-', 'blood_AB+', 'blood_AB-', 'blood_B+', 'blood_B-', 'blood_O+', 'blood_O-']


## STEP 7 — Multi-Label Encode Organs Donating

In [10]:
def clean_organs(organ_str):
    if pd.isna(organ_str): return []
    items = [i.strip() for i in str(organ_str).split(";")]
    return list(set(items))

df["organs_list"] = df["organs_donating"].apply(clean_organs)
mlb_organs = MultiLabelBinarizer()
organ_ohe = pd.DataFrame(mlb_organs.fit_transform(df["organs_list"]),
                         columns=mlb_organs.classes_, index=df.index)
df = pd.concat([df.drop(columns=["organs_list"]), organ_ohe], axis=1)
print(f"[Step 7] Multi-label encoded organs_donating → {list(mlb_organs.classes_)}")

[Step 7] Multi-label encoded organs_donating → ['Blood', 'Bone Marrow', 'Cornea', 'Heart', 'Kidney', 'Liver', 'Lung', 'Pancreas', 'Plasma', 'Platelet', 'Skin']


## STEP 8 — Encode Medical Conditions (multi-label grouped)

In [11]:
condition_groups = {
    "Condition_Diabetes":       ["Diabetes", "Diabetic Nephropathy", "Diabetic Retinopathy"],
    "Condition_Hypertension":   ["Hypertension", "Portal Hypertension",
                                 "Pulmonary Hypertension", "Hypertensive Nephropathy",
                                 "Renal Artery Stenosis"],
    "Condition_Heart_Disease":  ["Heart Disease", "Heart Failure", "Cardiomyopathy",
                                 "Cardiac Arrhythmia", "Myocarditis", "Cardiomegaly"],
    "Condition_Asthma":         ["Asthma", "COPD", "Bronchiectasis", "Bronchitis",
                                 "Cystic Fibrosis", "Pulmonary Fibrosis",
                                 "Interstitial Lung Disease", "Restrictive Lung Disease",
                                 "Pulmonary Embolism", "Pneumoconiosis", "Pneumonia", "Sarcoidosis"],
}

for col, keywords in condition_groups.items():
    pattern = "|".join([re.escape(k) for k in keywords])
    df[col] = df["medical_conditions"].str.contains(pattern, na=False).astype(int)

print(f"[Step 8] Multi-label encoded medical_conditions → {list(condition_groups.keys())}")

[Step 8] Multi-label encoded medical_conditions → ['Condition_Diabetes', 'Condition_Hypertension', 'Condition_Heart_Disease', 'Condition_Asthma']


## STEP 9 — Encode Cause of Death (grouped categories)

In [13]:
def group_cause_of_death(cause):
    if pd.isna(cause) or str(cause).lower() in ["unknown","none","n/a","alive"]:
        return "Not_Applicable"
    cause = str(cause).lower()
    if any(w in cause for w in ["heart","cardiac","infarction","stroke","cardiomyopathy","arrhythmia"]):
        return "Cardiovascular"
    elif any(w in cause for w in ["brain","cerebral","aneurysm","intracranial","hemorrhage","encephalitis"]):
        return "Neurological"
    elif any(w in cause for w in ["accident","injury","trauma","drowning"]):
        return "Accident_Trauma"
    elif any(w in cause for w in ["failure","insufficiency","cirrhosis","nephropathy"]):
        return "Organ_Failure"
    elif any(w in cause for w in ["respiratory","pulmonary","copd","fibrosis","ards"]):
        return "Respiratory"
    elif any(w in cause for w in ["sepsis","septic","shock"]):
        return "Infection_Sepsis"
    elif any(w in cause for w in ["cancer","leukemia","tumor"]):
        return "Cancer"
    elif any(w in cause for w in ["diabetic","ketoacidosis"]):
        return "Metabolic"
    else:
        return "Other_Unknown"

df["cause_category"] = df["cause_of_death"].apply(group_cause_of_death)
cause_dummies = pd.get_dummies(df["cause_category"], prefix="Cause").astype(int)
df = pd.concat([df, cause_dummies], axis=1)
print(f"[Step 9] Encoded cause_of_death → cause_category + {list(cause_dummies.columns)}")

[Step 9] Encoded cause_of_death → cause_category + ['Cause_Accident_Trauma', 'Cause_Cancer', 'Cause_Cardiovascular', 'Cause_Infection_Sepsis', 'Cause_Metabolic', 'Cause_Neurological', 'Cause_Not_Applicable', 'Cause_Organ_Failure', 'Cause_Other_Unknown', 'Cause_Respiratory']


## STEP 10 — Encode Hospital (label encode)

In [14]:
hospital_codes = {h: i for i, h in enumerate(sorted(df["hospital_name"].unique()))}
df["hospital_encoded"] = df["hospital_name"].map(hospital_codes)
df.drop(columns=["hospital_name"], inplace=True)
print(f"[Step 10] Label encoded hospital_name → hospital_encoded ({len(hospital_codes)} hospitals)")

[Step 10] Label encoded hospital_name → hospital_encoded (203 hospitals)


## STEP 11 — One-Hot Encode City

In [15]:
city_dummies = pd.get_dummies(df["city"], prefix="City").astype(int)
df = pd.concat([df.drop(columns=["city"]), city_dummies], axis=1)
print(f"[Step 11] One-hot encoded city → {list(city_dummies.columns)}")

[Step 11] One-hot encoded city → ['City_Abbottabad', 'City_Bahawalpur', 'City_Dera Ghazi Khan', 'City_Faisalabad', 'City_Gujranwala', 'City_Hyderabad', 'City_Islamabad', 'City_Jhelum', 'City_Karachi', 'City_Lahore', 'City_Mardan', 'City_Mirpur AK', 'City_Multan', 'City_Okara', 'City_Peshawar', 'City_Quetta', 'City_Rawalpindi', 'City_Sahiwal', 'City_Sargodha', 'City_Sialkot']


## STEP 12 — Drop Raw Text Columns

In [16]:
drop_raw = ["medical_conditions", "cause_of_death", "organs_donating"]
df.drop(columns=[c for c in drop_raw if c in df.columns], inplace=True)
print("[Step 12] Dropped raw text columns (encoded versions kept)")

[Step 12] Dropped raw text columns (encoded versions kept)


## STEP 13 — Final Column Order & Summary

In [17]:
print(f"\n[Step 13] Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", list(df.columns))
print("\nMissing values after preprocessing:")
miss = df.isnull().sum()
print(miss[miss > 0] if miss.any() else "  None ✓")
print("\nData types:")
print(df.dtypes.value_counts())


[Step 13] Final dataset: 800 rows × 73 columns
Columns: ['age', 'hepatitis_status', 'alive/deceased', 'death_hour', 'death_day', 'death_month', 'harvest_time_hours', 'gender_Male', 'blood_A+', 'blood_A-', 'blood_AB+', 'blood_AB-', 'blood_B+', 'blood_B-', 'blood_O+', 'blood_O-', 'Blood', 'Bone Marrow', 'Cornea', 'Heart', 'Kidney', 'Liver', 'Lung', 'Pancreas', 'Plasma', 'Platelet', 'Skin', 'Condition_Diabetes', 'Condition_Hypertension', 'Condition_Heart_Disease', 'Condition_Asthma', 'cause_category', 'Cause_Accident_Trauma', 'Cause_Cancer', 'Cause_Cardiovascular', 'Cause_Infection_Sepsis', 'Cause_Metabolic', 'Cause_Neurological', 'Cause_Not_Applicable', 'Cause_Organ_Failure', 'Cause_Other_Unknown', 'Cause_Respiratory', 'Cause_Accident_Trauma', 'Cause_Cancer', 'Cause_Cardiovascular', 'Cause_Infection_Sepsis', 'Cause_Metabolic', 'Cause_Neurological', 'Cause_Not_Applicable', 'Cause_Organ_Failure', 'Cause_Other_Unknown', 'Cause_Respiratory', 'hospital_encoded', 'City_Abbottabad', 'City_Baha

## STEP 14 — Save Preprocessed CSV

In [18]:
df.to_csv("donor_preprocessed.csv", index=False)
print("[Step 14] Saved: donor_preprocessed.csv")

[Step 14] Saved: donor_preprocessed.csv


## STEP 15 — Save Professionally Formatted XLSX

In [20]:
raw_df = pd.read_csv(r"E:\final datset of opal ai\dataset\donors(opal).csv")
output_path = r"E:\final datset of opal ai\dataset\donor_preprocessed_FINAL.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Preprocessed_Data", index=False)
    raw_df.to_excel(writer, sheet_name="Raw_Data", index=False)

    ref_rows = []
    ref_rows.append(["Column", "Type", "Description"])
    ref_rows.append(["age", "int", "Age of donor"])
    ref_rows.append(["gender_Male", "binary 0/1", "1=Male, 0=Female"])
    ref_rows.append(["hepatitis_status", "binary 0/1", "1=Positive, 0=Negative/Unknown"])
    ref_rows.append(["alive/deceased", "binary 0/1", "1=Alive, 0=Deceased"])
    ref_rows.append(["death_hour", "int 0-23", "Hour of death (-1 if alive)"])
    ref_rows.append(["death_day", "int 0-6", "Day of week of death (0=Mon)"])
    ref_rows.append(["death_month", "int 1-12", "Month of death (-1 if alive)"])
    ref_rows.append(["harvest_time_hours", "float", "Hours from death to harvest (-1 if alive)"])
    ref_rows.append(["cause_category", "string", "Grouped cause of death category"])
    ref_rows.append(["hospital_encoded", "int", "Label encoded hospital name"])
    for bt in ["A+","A-","AB+","AB-","B+","B-","O+","O-"]:
        ref_rows.append([f"blood_{bt}", "binary 0/1", f"1 if blood type is {bt}"])
    for org in ["Blood","Bone Marrow","Cornea","Heart","Kidney","Liver","Lung","Pancreas","Plasma","Platelet","Skin"]:
        ref_rows.append([org, "binary 0/1", f"1 if donating {org}"])
    for cond in condition_groups.keys():
        ref_rows.append([cond, "binary 0/1", "1 if condition group present"])
    cause_cats = ["Accident_Trauma","Cancer","Cardiovascular","Infection_Sepsis",
                  "Metabolic","Neurological","Not_Applicable","Organ_Failure","Other_Unknown","Respiratory"]
    for cat in cause_cats:
        ref_rows.append([f"Cause_{cat}", "binary 0/1", f"1 if cause of death is {cat}"])
    for city in sorted(raw_df["city"].dropna().unique()):
        ref_rows.append([f"City_{city}", "binary 0/1", f"1 if donor is from {city}"])

    ref_df = pd.DataFrame(ref_rows[1:], columns=ref_rows[0])
    ref_df.to_excel(writer, sheet_name="Data_Dictionary", index=False)

# ── Apply formatting ──────────────────────────────────────
wb = load_workbook(output_path)
header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
header_fill  = PatternFill("solid", start_color="1F4E79")
alt_fill     = PatternFill("solid", start_color="D6E4F0")
center_align = Alignment(horizontal="center", vertical="center")
left_align   = Alignment(horizontal="left",   vertical="center")
thin_border  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin"))

def style_sheet(ws):
    for cell in ws[1]:
        cell.font = header_font; cell.fill = header_fill
        cell.alignment = center_align; cell.border = thin_border
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill = alt_fill if i % 2 == 0 else PatternFill()
        for cell in row:
            cell.fill = fill; cell.alignment = left_align
            cell.border = thin_border; cell.font = Font(name="Arial", size=9)
    for col in ws.columns:
        max_len = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 30)
    ws.freeze_panes = "A2"

for sn in wb.sheetnames:
    style_sheet(wb[sn])

ws_dict = wb["Data_Dictionary"]
ws_dict.column_dimensions["A"].width = 30
ws_dict.column_dimensions["B"].width = 18
ws_dict.column_dimensions["C"].width = 55

wb.save(output_path)
print(f"[Step 15] Saved: {output_path}  ✓")
print("\n=== DONOR PREPROCESSING COMPLETE ===")

[Step 15] Saved: E:\final datset of opal ai\dataset\donor_preprocessed_FINAL.xlsx  ✓

=== DONOR PREPROCESSING COMPLETE ===
